# Rover on Rough Terrain — Planning Challenge (C++ edition, 30 min)

A point rover must drive from `start` to `goal` across rough 2D terrain. The
terrain has two kinds of rough ground — **mud** and **slope** — and every cell is
drivable, just at different (hidden) costs. Some experts have driven across this
terrain; their paths are the only clue to what the ground actually costs.

Your job is to turn those expert demos into a **cost map** that a planner can use
to drive like they do.

### The pieces you're given

Everything is bundled in a `World` value called **`w`** (loaded once in Setup and
passed to every function). **Convention:** points are `(row, col)`, so a cost map
is indexed `cost_map[row][col]`.

- **`w.features`** — a `(H × W × 2)` tensor
  (`std::vector<std::vector<std::array<float,2>>>`); `w.features[r][c]` is
  `[mud, slope]` at that cell, each in `0..1`. This is the raw terrain; it does
  **not** tell you how costly mud or slope actually are.
- **`w.demos`** — a few expert `(row, col)` polylines, each between its *own*
  start/goal (not yours). No single one is the answer; together they reveal which
  terrain the experts go out of their way to avoid.
- **`w.start`, `w.goal`** — `(row, col)` endpoints; **`w.H`, `w.W`** — grid size.
- **`astar_plan(w, cost_map)`** — a complete planner (`planner_p1.hpp`), already
  written for you. Hand it a cost map, get a near-optimal `(row, col)` path. You do
  **not** write a planner.

### Your one task

Write **`cost_map_learned(w)`** (in `learn_p1.hpp`) → a cost map that, routed
through `astar_plan`, yields a path as terrain-cheap as the experts' — cheap enough
to **PASS** the hidden scorer. You only `%%writefile learn_p1.hpp`.

A **uniform** cost map won't pass: it ignores mud and slope, so the planner drives
straight through the expensive ground. You have to make the terrain the experts
avoid cost *more* than the terrain they drive through.

### Setup

In [ ]:
import os, subprocess, struct
import numpy as np
if not os.path.exists('field_p1.npy'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field_p1.npy').astype(np.float32)        # (H, W, 2)
demos = np.load('demos_p1.npy', allow_pickle=True)        # object array of (T,2)
meta  = np.load('meta_p1.npz')
H, W = field.shape[:2]

with open('world_p1.bin', 'wb') as f:
    f.write(struct.pack('<ii', H, W))
    f.write(np.ascontiguousarray(field, np.float32).tobytes())
    f.write(np.ascontiguousarray(meta['true_cost'], np.float64).tobytes())
    f.write(np.asarray(meta['start'], np.float64).tobytes())
    f.write(np.asarray(meta['goal'],  np.float64).tobytes())
    f.write(struct.pack('<dd', float(meta['optimal_cost']), float(meta['tol'])))
    f.write(struct.pack('<i', len(demos)))
    for d in demos:
        d = np.asarray(d, np.float64)
        f.write(struct.pack('<i', d.shape[0]))
        f.write(np.ascontiguousarray(d, np.float64).tobytes())

import sys
sys.path.insert(0, os.getcwd())
from viz_p1 import show_cpp, build_and_run

print('g++ version:',
      subprocess.run(['g++', '--version'], capture_output=True, text=True).stdout.splitlines()[0])
print(f'packed world_p1.bin: field {field.shape} | demos {len(demos)} '
      f'| start {meta["start"]} goal {meta["goal"]}')
# Compile + run the shipped smoke-test driver (needs no candidate code), so a bad
# clone or toolchain surfaces here rather than inside the TODO. It routes the
# planner on a uniform cost map, which ignores mud/slope -> expect a FAIL line:
# that's the hook your learned cost has to beat.
_chk = subprocess.run(['g++', '-O2', '-std=c++17', 'main_run_baseline_p1.cpp', '-o', '_chk_p1'],
                      capture_output=True, text=True)
print('toolchain + shipped sources:', _chk.stderr.strip() or 'OK')
if _chk.returncode == 0:
    print(subprocess.run(['./_chk_p1'], capture_output=True, text=True).stdout)

In [ ]:
%%writefile learn_p1.hpp
// ---- TODO: learn a (H x W) cost map from the demos -----------------------
// Make terrain the experts drive through cheap, and terrain they detour around
// costly, so astar_plan routes like them. You only need the RANKING right, not
// the true cost numerically.
// (e.g. count w.demos visitation, or weight up mud/slope -- your call.)
//
// API (all in w; points are (row,col); do not edit the plumbing):
//   w.features[r][c]      : {mud, slope} at cell (r,c) (float[2])
//   w.demos               : vector<Path>, expert (row,col) polylines (q.r, q.c)
//   w.start, w.goal       : Vec2{r, c}; w.H, w.W : grid size
//   MUD, SLOPE            : channel indices 0, 1
//   Grid = vector<vector<double>> (index [row][col]); Path = vector<Vec2>
//   astar_plan(w, cm)     : -> Path, near-optimal (row,col) path under Grid cm
// The run cell routes your cost map and prints the PASS/FAIL line for you.
#pragma once
#include "metric_p1.hpp"
#include "terrain_p1.hpp"

inline Grid cost_map_learned(const World& w) {
    Grid cost_map(w.H, std::vector<double>(w.W, 1.0));  // uniform start -> FAILs
    // TODO: use w.demos to make mud/slope terrain cost more than clear ground.
    return cost_map;
}

### Run with your learned cost &mdash; goal: `PASS`

In [ ]:
build_and_run('main_run_p1.cpp', [('run_cost_p1.bin', 'learned: planned path over your learned cost map'), ('run_true_p1.bin', 'learned: planned path over the true cost map')])

### Discussion (optional — no code needed)

Be ready to talk through: where your learned cost generalizes badly; A* vs.
sampling vs. trajectory-opt and what each guarantees; the planner's Euclidean
heuristic (admissible here?); 8- vs 4-connectivity and resolution tradeoffs;
`float` vs `double` / cache locality in C++; a safety margin around rocks;
replanning in a 50 Hz loop.